In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from catboost import CatBoostRegressor, CatBoostClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif, f_regression

# Data loading

In [2]:

results_calculation = {}
ignore_filters = ['ts_gps_source','ts_gps_destination']
# Transform categorical features
categorical_features = [
  "Source",
  "Destination",
  "MCS",
  "Sub_channels",
  "channel_bandwidth"
]


In [3]:
def preprocess_nan_values(dataFrame,filters = []):
    """Replace NaN in categorical columns with empty string and maintain proper dtypes"""
   # Convert back to categorical type
    dataFrame = dataFrame.copy()
        

    for col in dataFrame.columns:
        if col in filters:
            dataFrame[col] = dataFrame[col].fillna('Unknown').astype(str)
        else:
            if pd.api.types.is_datetime64_any_dtype(dataFrame[col]):
                # Convert datetime to float (timestamp in seconds)
                dataFrame[col] = dataFrame[col].astype('int64') / 1e9
            else:
                dataFrame[col] = dataFrame[col].astype(float)
    
    return dataFrame



def preprocess_category_values(dataFrame,filters = []):
   # Convert back to categorical type
    dataFrame = dataFrame.copy()
        

    for col in dataFrame.columns:
        if col in filters:
            dataFrame[col] = dataFrame[col].fillna('Unknown').astype('category')
        else:
            if pd.api.types.is_datetime64_any_dtype(dataFrame[col]):
                dataFrame[col] = dataFrame[col].astype('int64') / 1e9
            else:
                dataFrame[col] = dataFrame[col].astype(float)
    
    return dataFrame

In [4]:
cellular_df = preprocess_nan_values(pd.read_csv('cellular_df.csv'),ignore_filters)
cellular_df.drop(columns=ignore_filters,inplace=True)
cellular_df = preprocess_category_values(cellular_df,categorical_features)

# Data preparation

In [5]:
# Split inputs and targets

X = cellular_df.drop(columns=['throughput'])
y = cellular_df['throughput']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)



# oe = OrdinalEncoder()
# train_inputs[categorical_features] = oe.fit_transform(train_inputs[categorical_features])
# test_inputs[categorical_features] = oe.transform(test_inputs[categorical_features])

# # Missing value imputation
# train_inputs.fillna(0, inplace=True)
# test_inputs.fillna(0, inplace=True)

# Prediction algorithm using Random Forest

In [6]:

# Create algorithm
rf = RandomForestRegressor(
     n_estimators=100,
    max_depth=4,
    verbose=1000
)

# Train
rf.fit(X_train, y_train)

# Validation
val_predictions = rf.predict(X_test)



building tree 1 of 100
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    1.5s
building tree 2 of 100
[Parallel(n_jobs=1)]: Done   2 tasks      | elapsed:    2.9s
building tree 3 of 100
[Parallel(n_jobs=1)]: Done   3 tasks      | elapsed:    4.7s
building tree 4 of 100
[Parallel(n_jobs=1)]: Done   4 tasks      | elapsed:    6.3s
building tree 5 of 100
[Parallel(n_jobs=1)]: Done   5 tasks      | elapsed:    7.7s
building tree 6 of 100
[Parallel(n_jobs=1)]: Done   6 tasks      | elapsed:    9.1s
building tree 7 of 100
[Parallel(n_jobs=1)]: Done   7 tasks      | elapsed:   10.6s
building tree 8 of 100
[Parallel(n_jobs=1)]: Done   8 tasks      | elapsed:   12.0s
building tree 9 of 100
[Parallel(n_jobs=1)]: Done   9 tasks      | elapsed:   13.4s
building tree 10 of 100
[Parallel(n_jobs=1)]: Done  10 tasks      | elapsed:   14.8s
building tree 11 of 100
[Parallel(n_jobs=1)]: Done  11 tasks      | elapsed:   16.2s
building tree 12 of 100
[Parallel(n_jobs=1)]: Done  12 tasks      | elapse

# Compute error metric for Random Forest

In [7]:

r2 = r2_score(y_test, val_predictions)
rmse = np.sqrt(mean_squared_error(y_test, val_predictions))
mae = mean_absolute_error(y_test, val_predictions)
print(f"R²: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

results_calculation['RF'] = {'r2':r2, 'rmse': rmse, 'mae':mae}


R²: 0.9982
RMSE: 8.0963
MAE: 4.7734


# Prediction using XGBoost

In [8]:

model = XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42,
        enable_categorical=True
    )

model.fit(X_train, y_train)

y_pred_xg = model.predict(X_test)



In [9]:

r2_xg = r2_score(y_test, y_pred_xg)
rmse_xg = np.sqrt(mean_squared_error(y_test, y_pred_xg))
mae_xg = mean_absolute_error(y_test, y_pred_xg)
print(f"R²: {r2_xg:.4f}")
print(f"RMSE: {rmse_xg:.3f}")
print(f"MAE: {mae_xg:.4f}")

results_calculation['XGBoost'] = {'r2':r2_xg, 'rmse': rmse_xg, 'mae':mae_xg}


R²: 0.9996
RMSE: 3.894
MAE: 0.3296


# Cat Boost Algorithm Prediction

In [10]:


cat_features_indices = [X_train.columns.get_loc(col) for col in categorical_features]

from catboost import Pool

#POOLING
train_pool = Pool(
    data=preprocess_nan_values(X_train,categorical_features),
    label=y_train,
    cat_features=categorical_features
)

test_pool = Pool(
    data=preprocess_nan_values(X_test,categorical_features),
    label=y_test,
    cat_features=categorical_features
)

# Model for CatBoostAlgorithm 
model_cat = CatBoostRegressor(
    iterations=4000,
    learning_rate=0.05,
    depth=4,
    verbose=1000,
    cat_features=cat_features_indices,
    random_seed=42,
    early_stopping_rounds=10,
    l2_leaf_reg=10,
    loss_function='RMSE',
    allow_writing_files=False
)



model_cat.fit(train_pool,eval_set=test_pool)

pred_cat = model_cat.predict(test_pool)







0:	learn: 182.8024640	test: 182.7074537	best: 182.7074537 (0)	total: 112ms	remaining: 7m 28s
Stopped by overfitting detector  (10 iterations wait)

bestTest = 4.01962287
bestIteration = 450

Shrink model to first 451 iterations.


In [11]:
r2_cat = r2_score(y_test, pred_cat)
rmse_cat = np.sqrt(mean_squared_error(y_test, pred_cat))
mae_cat = mean_absolute_error(y_test, pred_cat)



print(f"R²: {r2_cat:.4f}")
print(f"RMSE: {rmse_cat:.4f}")
print(f"MAE: {mae_cat:.4f}")

results_calculation['CatBoost'] = {'r2':r2_cat, 'rmse':  rmse_cat, 'mae':mae_cat}


R²: 0.9996
RMSE: 4.0196
MAE: 0.7565


In [12]:
import plotly.graph_objects as go

model_names = list(results_calculation.keys())
r2_values = [results_calculation[model]['r2'] for model in model_names]
rmse_values = [results_calculation[model]['rmse'] for model in model_names]
mae_values = [results_calculation[model]['mae'] for model in model_names]

# Creating the bar chart
fig = go.Figure()

# Adding R-Square (R²) bars
fig.add_trace(go.Bar(
    x=model_names,
    y=r2_values,
    name='R² Error',
    marker_color='blue'
))

# Adding Root Mean Squared Error (RMSE) bars
fig.add_trace(go.Bar(
    x=model_names,
    y=rmse_values,
    name='RMSE',
    marker_color='green'
))

# Adding MAE bars
fig.add_trace(go.Bar(
    x=model_names,
    y=mae_values,
    name='MAE',
    marker_color='red'
))

# Updating layout
fig.update_layout(
    title='Comparison of Model Performance Metrics',
    xaxis_title='Models',
    yaxis_title='Metric Values',
    barmode='group',
    legend_title='Metrics',
    template='plotly_white'
)

# Show the plot
fig.show()